# Urdu Question Generation - Kaggle training run

From-scratch **2-layer BiLSTM encoder-decoder with Bahdanau attention**. Input: an Urdu
sentence with the answer wrapped in `<ans> ... </ans>`. Output: the question that span
answers.

Each step below calls a module in the repo's `src/` package, so the code is identical
locally and here and every step can be pointed to in the viva.

**Settings panel (right): Accelerator = `GPU T4 x2`  |  Internet = `On`.**
(The P100 is compute-capability 6.0 and Kaggle's current PyTorch has no kernels for it;
the T4 is 7.5 and works. The code uses one T4.)

In [ ]:
# 1. Setup: get the code, install the few missing deps, confirm the GPU runs CUDA kernels.
%cd /kaggle/working
!rm -rf urdu-question-generation
!git clone --depth 1 https://github.com/Hanzala-12/urdu-question-generation.git
%cd urdu-question-generation
!pip -q install sentencepiece sacrebleu rouge-score
import torch
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU!')
print('CUDA kernels OK:', (torch.ones(3, device='cuda') * 2).tolist())

In [ ]:
# 2. Task 1 - data preparation. Downloads UQA + Wiki-UQA (Internet must be On),
#    keeps answerable rows, finds the answer sentence, wraps the span, writes TSVs.
!python -m src.data_prep

In [ ]:
# 3. Task 2 - train the 8k SentencePiece unigram tokenizer on the training text.
!python -m src.spm_train

In [ ]:
# 4. Debug gate - 10k pairs, 1 epoch. The loss MUST fall; if not, stop and fix the bug.
!python -m src.train --debug

In [ ]:
# 5. Task 3 - full training. ~1-2 h on a T4 (batch 64, 15 epochs).
#    Saves artifacts/best.pt (best val loss) and results/loss_log.csv + loss_curve.png.
!python -m src.train --epochs 15 --batch-size 64

In [ ]:
# 6. Task 4 - evaluation on UQA-valid and Wiki-UQA, greedy + beam.
#    Writes results/metrics.json, samples.tsv, tables.md, figures/attention.png.
!python -m src.evaluate --split both --beam-max 3000
import json
print(json.dumps(json.load(open('results/metrics.json')), indent=2, ensure_ascii=False))

In [ ]:
# 7. Bundle artifacts/ + results/ into one zip to download back into the local repo.
import shutil
shutil.copytree('artifacts', '/kaggle/working/outputs/artifacts', dirs_exist_ok=True)
shutil.copytree('results', '/kaggle/working/outputs/results', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/outputs')
print('done -> /kaggle/working/outputs.zip')
!ls -lhR /kaggle/working/outputs